In [1]:
import os 
import pandas as pd 
import numpy as np
import tensorflow as tf
import keras 
from keras import layers, models, Input
from keras.optimizers import Adam
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from tensorflow.keras import regularizers
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, classification_report
import sqlite3

2025-12-05 03:05:37.584572: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.1 SSE4.2 AVX AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-12-05 03:05:37.899114: I tensorflow/core/util/port.cc:104] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
TIMEFRAME = 30
BATCH_SIZE = 64
STRIDE = 5
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

Num GPUs Available:  0


In [3]:

conn = sqlite3.connect('/home/viktor/ai2/training_data_v0.db')
#dataset = pd.read_csv(filepath_or_buffer='/home/viktor/ai2/cleaned_data.csv')
dataset = pd.read_sql("SELECT * FROM training_data_v0", conn)
conn.close()

n_samples = len(dataset)
train_end = int(0.80 * n_samples)
df_data = dataset[:train_end]
train_set = dataset[train_end:]




In [ ]:
def merge_chroma_bins(df):
    """
    Takes a dataframe with 24 columns.
    Adds the Bass (0-11) and Treble (12-23) bins together.
    Returns a dataframe with 12 columns.
    """
    # Slice the first 12 (Bass) and last 12 (Treble)
    bass = df.iloc[:, 0:12]
    treble = df.iloc[:, 12:24]

    # Sum them together. 
    # We use .values on treble to ignore column name mismatches
    merged_val = bass.values + treble.values

    #normalize the dataset so that it resembels output from librosa audioprocessing
    max_vals = np.max(merged_val, axis=1, keepdims=True)
    normalized_values = merged_val / (max_vals + 1e-9)
    return pd.DataFrame(normalized_values, index=df.index)

In [ ]:
#create two sets of data on for training and validation other for testing
x_data = df_data.drop(columns=['key','quality','timestamp'])
x_data = merge_chroma_bins(x_data)
y_key_data = df_data['key']
y_quality_data = df_data['quality']

data_x_test = train_set.drop(columns=['key','quality','timestamp'])
data_x_test = merge_chroma_bins(data_x_test)
data_key_test = train_set['key']
data_quality_test = train_set['quality']




#split training data into training and validation data
x_train, x_val, y_key_train, y_key_val, y_quality_train, y_quality_val = train_test_split(x_data, 
    y_key_data, 
    y_quality_data, 
    test_size=0.15, 
    shuffle=False)

#create sliding windows so that we give the model more context for each predicton
x_train = tf.keras.utils.timeseries_dataset_from_array(
    data= x_train,
    targets= None,
    sequence_length=TIMEFRAME,#size of the sliding window
    sequence_stride=STRIDE,#increase stride during training to prevent overfitting and increase training speed 
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=42
)

y_key_train = tf.keras.utils.timeseries_dataset_from_array(
    data= y_key_train[TIMEFRAME-1:],
    targets= None,
    sequence_length=1,
    sequence_stride=STRIDE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=42
)

y_quality_train = tf.keras.utils.timeseries_dataset_from_array(
    data= y_quality_train[TIMEFRAME-1:],
    targets= None,
    sequence_length=1,
    sequence_stride=STRIDE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=42
)

#validation data
x_val = tf.keras.utils.timeseries_dataset_from_array(
    data= x_val,
    targets= None,
    sequence_length=TIMEFRAME,
    sequence_stride=STRIDE,
    batch_size=BATCH_SIZE,
    shuffle=False,
    seed=42
)

y_key_val = tf.keras.utils.timeseries_dataset_from_array(
    data= y_key_val[TIMEFRAME-1:],
    targets= None,
    sequence_length=1,
    sequence_stride=STRIDE,
    batch_size=BATCH_SIZE,
    shuffle=False,
    seed=42
)

y_quality_val = tf.keras.utils.timeseries_dataset_from_array(
    data= y_quality_val[TIMEFRAME-1:],
    targets= None,
    sequence_length=1,
    sequence_stride=STRIDE,
    batch_size=BATCH_SIZE,
    shuffle=False,
    seed=42
)

#testdata
x_test = tf.keras.utils.timeseries_dataset_from_array(
    data= data_x_test,
    targets= None,
    sequence_length=TIMEFRAME,
    sequence_stride=1,
    batch_size=BATCH_SIZE,
    shuffle=False,
    seed=42
)

key_test = tf.keras.utils.timeseries_dataset_from_array(
    data= data_key_test[TIMEFRAME-1:],
    targets= None,
    sequence_length=1,
    sequence_stride=1,
    batch_size=BATCH_SIZE,
    shuffle=False,
    seed=42
)

quality_test = tf.keras.utils.timeseries_dataset_from_array(
    data= data_quality_test[TIMEFRAME-1:],
    targets= None,
    sequence_length=1,
    sequence_stride=1,
    batch_size=BATCH_SIZE,
    shuffle=False,
    seed=42
)


2025-12-05 03:06:15.216838: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.1 SSE4.2 AVX AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [6]:
def y_mapping_reshaped(x, y_key, y_qual):
    # Squeeze removes the time dimension from Y since it's just 1 step now
    return x, {'key': tf.squeeze(y_key, axis=1), 'quality': tf.squeeze(y_qual, axis=1)}

In [7]:
train_zipped = tf.data.Dataset.zip((x_train, y_key_train, y_quality_train))
validation_zipped = tf.data.Dataset.zip((x_val, y_key_val, y_quality_val))
test_zipped = tf.data.Dataset.zip((x_test,key_test,quality_test))
 
train_data = train_zipped.map(y_mapping_reshaped, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)
validation_data = validation_zipped.map(y_mapping_reshaped, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)
test_data = test_zipped.map(y_mapping_reshaped).prefetch(tf.data.AUTOTUNE)

In [8]:
inputs = Input(shape= (TIMEFRAME,12))
output = layers.GaussianNoise(0.05)(inputs)
output = layers.Conv1D(filters=32, kernel_size=3,strides=1,padding='same',activation='relu')(output)
output = layers.BatchNormalization()(output)
output = layers.Conv1D(64, kernel_size=5, padding='same', activation='relu')(output)
output = layers.BatchNormalization()(output)
output = layers.MaxPooling1D(pool_size=2)(output)
output = layers.Dropout(0.4)(output)
output = layers.Bidirectional(layers.GRU(32, return_sequences=True, dropout=0.3, unroll=True))(output)
output = layers.Bidirectional(layers.GRU(64, return_sequences=False, dropout=0.3, unroll=True))(output)
output = layers.Dropout(0.4)(output)


keyInput= layers.Dense(128,activation='relu')(output)
key= layers.Dense(14,activation= 'softmax',name='key',kernel_regularizer=regularizers.l2(0.001))(keyInput)

shapeInput = layers.Dense(64,activation='relu')(output)
quality = layers.Dense(4,activation='softmax',name='quality',kernel_regularizer=regularizers.l2(0.001))(shapeInput)

model3x = models.Model(inputs=inputs, outputs = [key, quality])
model3x.compile(optimizer=Adam(learning_rate=0.001),
              loss= 'sparse_categorical_crossentropy',
              metrics=['accuracy'],
              )


In [ ]:


lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss', 
    factor=0.2,       # Reduce LR by 5x (multiply by 0.2)
    patience=2,       # Wait 2 epochs with no improvement
    min_lr=0.00001,   # Don't go below this
    verbose=1
)

model3x.fit(
        train_data,
        validation_data=validation_data,
        epochs= 10,
        callbacks=[
        tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True), lr_scheduler
        ]
)



Epoch 1/10
3013/3013 [==============================] - 249s 78ms/step - loss: 1.8492 - key_loss: 1.1269 - quality_loss: 0.6996 - key_accuracy: 0.6721 - quality_accuracy: 0.7258 - val_loss: 1.6810 - val_key_loss: 1.0080 - val_quality_loss: 0.6573 - val_key_accuracy: 0.7257 - val_quality_accuracy: 0.7237 - lr: 0.0010
Epoch 2/10
3013/3013 [==============================] - 226s 75ms/step - loss: 1.6081 - key_loss: 0.9842 - quality_loss: 0.6105 - key_accuracy: 0.7098 - quality_accuracy: 0.7639 - val_loss: 1.6258 - val_key_loss: 0.9800 - val_quality_loss: 0.6339 - val_key_accuracy: 0.7342 - val_quality_accuracy: 0.7428 - lr: 0.0010
Epoch 3/10
3013/3013 [==============================] - 234s 78ms/step - loss: 1.5350 - key_loss: 0.9421 - quality_loss: 0.5816 - key_accuracy: 0.7198 - quality_accuracy: 0.7758 - val_loss: 1.5637 - val_key_loss: 0.9423 - val_quality_loss: 0.6109 - val_key_accuracy: 0.7381 - val_quality_accuracy: 0.7606 - lr: 0.0010
Epoch 4/10
3013/3013 [========================

In [ ]:
#evaluate model
model3x.summary()

model3x.evaluate(
    test_data
)

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_1 (InputLayer)           [(None, 30, 12)]     0           []                               
                                                                                                  
 gaussian_noise (GaussianNoise)  (None, 30, 12)      0           ['input_1[0][0]']                
                                                                                                  
 conv1d (Conv1D)                (None, 30, 32)       1184        ['gaussian_noise[0][0]']         
                                                                                                  
 batch_normalization (BatchNorm  (None, 30, 32)      128         ['conv1d[0][0]']                 
 alization)                                                                                   

[1.8744784593582153,
 1.1943128108978271,
 0.6713343858718872,
 0.6833125948905945,
 0.753711462020874]

In [ ]:
#wrapp model so that we can save it with preprocessing steps included
class ModelWrapper(tf.Module):

    def __init__(self, model, timeframe=30):
        super().__init__()
        self.model = model
        self.timeframe = timeframe
    
    @tf.function(input_signature=[tf.TensorSpec(shape=[None, 12], dtype=tf.float32)])

    def __call__(self, features):
        # 1. Create Padding (Repeat the first row 39 times)
        # We take features[0], expand dims to (1, 24), and tile it
        first_frame = tf.expand_dims(features[0], 0)
        padding = tf.tile(first_frame, [self.timeframe - 1, 1])
        
        # 2. Stick padding on top
        padded_features = tf.concat([padding, features], axis=0)
        
        # 3. Frame it
        windows = tf.signal.frame(padded_features, frame_length=self.timeframe, frame_step=1, axis=0)
        
        # 4. Predict
        predictions = self.model(windows)
        
        return {"key_probs": predictions[0], "quality_probs": predictions[1]}

In [12]:
wrapped_model = ModelWrapper(model3x, timeframe=TIMEFRAME)

In [ ]:
#testing that it works
test_input = tf.random.normal((100, 12))
output = wrapped_model(test_input)
print("Key probs shape:", output["key_probs"].shape)
print("Quality probs shape:", output["quality_probs"].shape)

Key probs shape: (100, 14)
Quality probs shape: (100, 4)


In [15]:
tf.saved_model.save(
    wrapped_model, 
    'HarmonAi_v1',
    signatures={'serving_default': wrapped_model.__call__}
)
2.0116

INFO:tensorflow:Assets written to: HarmonAi_v1/assets


INFO:tensorflow:Assets written to: HarmonAi_v1/assets


2.0116